# 05 — Ablation Analysis

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/nested_ablation_*.csv`, `../outputs/tables/state_trait_glm.csv`

**Description:**
- Nested ablation: content vs engagement vs instability vs constriction
- Four text feature categories:
  - **Content (C):** PCA on sentence-transformer embeddings — *what* people write
  - **Engagement (E):** word count features — *how much* people write
  - **Instability (I):** cosine distance to person centroid — *how different* from their average
  - **Constriction (K):** type-token ratio, root TTR — *how restricted* their vocabulary
- Incremental utility: does text add value beyond numeric crisis ratings?
- CV-safe instability (centroids from training folds only)
- State vs trait decomposition via cluster-robust GLM

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import statsmodels.api as sm

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
TEXT_COL = "pm_day_text"
CRISIS_COL = "crisis_PM_from_full"

N_PCS = 20
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day)
print("Loaded data:", pm_day.shape)

# Prepare feature blocks
X_eng = pm_day[["log1p_wc", "wc_le1"]].values.astype(float)
X_crisis = pm_day[[CRISIS_COL]].astype(float).values

# Constriction features — impute NaN (minimal text) with column median
# NaN occurs for responses with <=1 word where TTR is undefined
X_constrict_df = pm_day[["ttr", "root_ttr"]].copy()
X_constrict_df = X_constrict_df.fillna(X_constrict_df.median())
X_constrict = X_constrict_df.values.astype(float)

groups = pm_day[PID_COL].astype(str).values

mask_ok = (
    np.isfinite(X_crisis).all(axis=1) &
    np.isfinite(X_eng).all(axis=1) &
    np.isfinite(X_constrict).all(axis=1)
)
print(f"Rows retained: {mask_ok.sum()} / {len(pm_day)}")
print(f"Constriction features: ttr, root_ttr (NaN imputed with median)")

In [ ]:
# =========================
# HELPERS
# =========================
def l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)


def fit_predict_logit(Xtr, Xte, ytr, C=1.0, random_state=0):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(
            penalty="l2", C=C, solver="liblinear",
            max_iter=5000, random_state=random_state
        ))
    ])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

In [ ]:
# =========================
# NESTED ABLATION (with CV-safe instability, 4 feature categories)
# =========================
def run_nested_ablation(X_text, X_eng, X_constrict, X_crisis, y, groups, n_pcs=20):
    gkf = GroupKFold(n_splits=5)
    X_text_norm = l2_normalize_rows(X_text)

    # Model naming convention:
    #   C = Content (PCA), E = Engagement (wc), I = Instability (cosdist),
    #   K = Constriction (TTR), N = Numeric baseline
    model_names = [
        # Text-only models (building up)
        "C",          # content only
        "CE",         # + engagement
        "CK",         # + constriction (skipping engagement)
        "CEK",        # content + engagement + constriction
        "CEI",        # content + engagement + instability (no constriction)
        "CEIK",       # all text features
        # Numeric baseline models (building up)
        "N",          # numeric only
        "NC",         # + content
        "NCK",        # + content + constriction
        "NCE",        # + content + engagement
        "NCEK",       # + content + engagement + constriction
        "NCEI",       # + content + engagement + instability
        "NCEIK",      # numeric + all text features
    ]
    oof = {k: np.full(len(y), np.nan) for k in model_names}

    for tr, te in gkf.split(X_text, y, groups):
        # PCA on text (fit on train only)
        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])

        # CV-safe instability (centroids from train only)
        tr_idx, te_idx = np.array(tr), np.array(te)
        train_groups = groups[tr_idx]
        global_centroid = X_text_norm[tr_idx].mean(axis=0)
        global_centroid /= max(np.linalg.norm(global_centroid), 1e-12)

        pid_to_centroid = {}
        for pid in np.unique(train_groups):
            rows = X_text_norm[tr_idx[train_groups == pid]]
            c = rows.mean(axis=0)
            c /= max(np.linalg.norm(c), 1e-12)
            pid_to_centroid[pid] = c

        instab_tr = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)
                              for idx in tr_idx]).reshape(-1, 1)
        instab_te = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)
                              for idx in te_idx]).reshape(-1, 1)

        # Shorthand for feature matrices
        Etr, Ete = X_eng[tr], X_eng[te]
        Ktr, Kte = X_constrict[tr], X_constrict[te]
        Ntr, Nte = X_crisis[tr], X_crisis[te]
        Itr, Ite = instab_tr, instab_te

        # --- Text-only models ---
        oof["C"][te] = fit_predict_logit(Ttr, Tte, y[tr])
        oof["CE"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr]), np.hstack([Tte, Ete]), y[tr])
        oof["CK"][te] = fit_predict_logit(
            np.hstack([Ttr, Ktr]), np.hstack([Tte, Kte]), y[tr])
        oof["CEK"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Ktr]), np.hstack([Tte, Ete, Kte]), y[tr])
        oof["CEI"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Itr]), np.hstack([Tte, Ete, Ite]), y[tr])
        oof["CEIK"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Itr, Ktr]), np.hstack([Tte, Ete, Ite, Kte]), y[tr])

        # --- Numeric baseline models ---
        oof["N"][te] = fit_predict_logit(Ntr, Nte, y[tr])
        oof["NC"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr]), np.hstack([Nte, Tte]), y[tr])
        oof["NCK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Ktr]), np.hstack([Nte, Tte, Kte]), y[tr])
        oof["NCE"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr]), np.hstack([Nte, Tte, Ete]), y[tr])
        oof["NCEK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Ktr]), np.hstack([Nte, Tte, Ete, Kte]), y[tr])
        oof["NCEI"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Itr]), np.hstack([Nte, Tte, Ete, Ite]), y[tr])
        oof["NCEIK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Itr, Ktr]), np.hstack([Nte, Tte, Ete, Ite, Kte]), y[tr])

    # Build results with readable labels
    label_map = {
        "C":     "Content",
        "CE":    "Content + Engagement",
        "CK":    "Content + Constriction",
        "CEK":   "Content + Engagement + Constriction",
        "CEI":   "Content + Engagement + Instability",
        "CEIK":  "Content + Engagement + Instability + Constriction",
        "N":     "Numeric only",
        "NC":    "Numeric + Content",
        "NCK":   "Numeric + Content + Constriction",
        "NCE":   "Numeric + Content + Engagement",
        "NCEK":  "Numeric + Content + Engagement + Constriction",
        "NCEI":  "Numeric + Content + Engagement + Instability",
        "NCEIK": "Numeric + All text features",
    }

    rows = []
    for name, p in oof.items():
        rows.append({
            "model": name,
            "label": label_map[name],
            "AUROC": roc_auc_score(y, p),
            "AUPRC": average_precision_score(y, p),
        })
    return pd.DataFrame(rows)

In [ ]:
# =========================
# RUN ABLATION FOR EACH OUTCOME
# =========================
for outcome in ["high_any_item_eq3", "moderate_total_ge2", "any_risk_total_gt0"]:
    y = pm_day[outcome].astype(int).values
    y_m = y[mask_ok]
    X_text_m = X_text[mask_ok]
    X_eng_m = X_eng[mask_ok]
    X_constrict_m = X_constrict[mask_ok]
    X_crisis_m = X_crisis[mask_ok]
    groups_m = groups[mask_ok]

    if y_m.sum() < 20:
        print(f"Skipping {outcome}: too few positives")
        continue

    print("")
    print("=" * 60)
    print(f"Outcome: {outcome} | N={len(y_m)} | pos={y_m.sum()} ({y_m.mean():.3f})")
    print("=" * 60)

    results = run_nested_ablation(
        X_text_m, X_eng_m, X_constrict_m, X_crisis_m, y_m, groups_m, n_pcs=N_PCS
    )

    # Display in two blocks for readability
    text_only = results[~results["model"].str.startswith("N")]
    with_numeric = results[results["model"].str.startswith("N")]

    print("")
    print("--- Text-only models ---")
    print(text_only[["model", "label", "AUROC", "AUPRC"]].to_string(index=False))
    print("")
    print("--- With numeric baseline ---")
    print(with_numeric[["model", "label", "AUROC", "AUPRC"]].to_string(index=False))

    out_path = os.path.join(OUT_DIR, f"nested_ablation_{outcome}.csv")
    results.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

In [ ]:
# =========================
# STATE vs TRAIT GLM (cluster-robust) --- this is for high suicide risk days
# =========================
# Now includes constriction (ttr, root_ttr) in the GLM
N_PCS_CONTENT = 5
OUTCOME_BIN = "high_any_item_eq3"
MECH_COLS = (
    ["log1p_wc", "wc_le1", "instability_cosdist", "ttr", "root_ttr"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

X_cols = []
for c in MECH_COLS:
    X_cols += [f"{c}_within", f"{c}_between"]

dfm = pm_day.copy()
y_glm = dfm[OUTCOME_BIN].astype(float).values
mask_glm = np.isfinite(y_glm)
for c in list(X_cols):
    if c in dfm.columns:
        mask_glm &= np.isfinite(dfm[c].values)
    else:
        print(f"Warning: {c} not in data, skipping from GLM")
        X_cols = [x for x in X_cols if x != c]

dfm = dfm.loc[mask_glm].copy()
y_glm = dfm[OUTCOME_BIN].astype(int).values
X_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")
groups_glm = dfm[PID_COL].astype(str).values

glm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())
res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})

print("")
print("=== STATE vs TRAIT (cluster-robust GLM) ===")
print(f"Outcome: {OUTCOME_BIN} | N={len(dfm)} | participants={dfm[PID_COL].nunique()}")
print(res.summary())

# Save table
rows = []
for c in MECH_COLS:
    for part in ["within", "between"]:
        name = f"{c}_{part}"
        if name in res.params.index:
            rows.append({
                "feature": c, "component": part,
                "beta": float(res.params[name]),
                "SE": float(res.bse[name]),
                "z": float(res.tvalues[name]),
                "p": float(res.pvalues[name]),
                "OR": float(np.exp(res.params[name])),
                "CI_lo": float(res.conf_int().loc[name, 0]),
                "CI_hi": float(res.conf_int().loc[name, 1]),
            })

tab = pd.DataFrame(rows)
tab_path = os.path.join(OUT_DIR, "state_trait_glm.csv")
tab.to_csv(tab_path, index=False)
print(f"Saved: {tab_path}")

# Highlight constriction findings
print("")
print("--- Constriction effects ---")
constrict_rows = tab[tab["feature"].isin(["ttr", "root_ttr"])]
for _, row in constrict_rows.iterrows():
    sig = "*" if row["p"] < .05 else "~" if row["p"] < .10 else ""
    print(f"  {row['feature']}_{row['component']}: b={row['beta']:.3f} z={row['z']:.3f} p={row['p']:.3f} {sig}")

In [ ]:
# =========================
# STATE vs TRAIT GLM — TRIMMED (cluster-robust)
# =========================
# Trimmed predictor set: removed wc_le1 (redundant with log1p_wc); multicollinearity
# and ttr (root_ttr preferred — corrects for length dependence)
N_PCS_CONTENT = 5
OUTCOME_BIN = "high_any_item_eq3"
MECH_COLS = (
    ["log1p_wc", "instability_cosdist", "root_ttr"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

X_cols = []
for c in MECH_COLS:
    X_cols += [f"{c}_within", f"{c}_between"]

dfm = pm_day.copy()
y_glm = dfm[OUTCOME_BIN].astype(float).values
mask_glm = np.isfinite(y_glm)
for c in list(X_cols):
    if c in dfm.columns:
        mask_glm &= np.isfinite(dfm[c].values)
    else:
        print(f"Warning: {c} not in data, skipping from GLM")
        X_cols = [x for x in X_cols if x != c]

dfm = dfm.loc[mask_glm].copy()
y_glm = dfm[OUTCOME_BIN].astype(int).values
X_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")
groups_glm = dfm[PID_COL].astype(str).values

print(f"Predictors: {X_cols}")
print(f"N = {len(dfm)} | participants = {dfm[PID_COL].nunique()}")
print(f"Positive cases: {y_glm.sum()} ({y_glm.mean():.3f})")
print("")

glm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())
res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})

print("=== STATE vs TRAIT GLM (trimmed, cluster-robust) ===")
print(res.summary())

# Save table
rows = []
for c in MECH_COLS:
    for part in ["within", "between"]:
        name = f"{c}_{part}"
        if name in res.params.index:
            rows.append({
                "feature": c, "component": part,
                "beta": float(res.params[name]),
                "SE": float(res.bse[name]),
                "z": float(res.tvalues[name]),
                "p": float(res.pvalues[name]),
                "OR": float(np.exp(res.params[name])),
                "CI_lo": float(res.conf_int().loc[name, 0]),
                "CI_hi": float(res.conf_int().loc[name, 1]),
            })

tab = pd.DataFrame(rows)
tab_path = os.path.join(OUT_DIR, "state_trait_glm.csv")
tab.to_csv(tab_path, index=False)
print(f"Saved: {tab_path}")

# Highlight key findings
print("")
print("--- Significant / trending effects ---")
for _, row in tab.iterrows():
    if row["p"] < .10:
        sig = "**" if row["p"] < .05 else "*" if row["p"] < .10 else ""
        print(f"  {row['feature']}_{row['component']}: b={row['beta']:.3f} z={row['z']:.3f} p={row['p']:.3f} OR={row['OR']:.3f} {sig}")

In [ ]:
# still seems like there is multicollinearity issues check VIF

from statsmodels.stats.outliers_influence import variance_inflation_factor

X_check = dfm[X_cols].astype(float).values
vif = pd.DataFrame({
    "feature": X_cols,
    "VIF": [variance_inflation_factor(X_check, i) for i in range(X_check.shape[1])]
})
print(vif.to_string(index=False))
print(f"\nEvents per variable: {y_glm.sum()} events / {len(X_cols)} predictors = {y_glm.sum()/len(X_cols):.1f}")

In [ ]:
## look at correlations with VIF to investigate

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- pick the columns you want to diagnose ---
cols = [
    "log1p_wc_within","log1p_wc_between",
    "instability_cosdist_within","instability_cosdist_between",
    "root_ttr_within","root_ttr_between",
    "PC1_within","PC1_between",
    "PC2_within","PC2_between",
    "PC3_within","PC3_between",
    "PC4_within","PC4_between",
    "PC5_within","PC5_between",
]

dfx = pm_day.copy()

# keep only columns that exist
cols = [c for c in cols if c in dfx.columns]

# numeric + complete cases
X = dfx[cols].apply(pd.to_numeric, errors="coerce")
X = X.dropna()

print("N rows used:", len(X))
print("Columns:", X.columns.tolist())

# -------------------------
# 1) Correlations (quick)
# -------------------------
corr = X.corr()
print("\n=== Top absolute correlations (|r|) ===")
abs_corr = corr.abs()
np.fill_diagonal(abs_corr.values, np.nan)

top = (
    abs_corr.stack()
    .sort_values(ascending=False)
    .head(20)
)

for (a, b), r in top.items():
    print(f"{a}  ~  {b}:  r={corr.loc[a,b]:.3f}")

# -------------------------
# 2) VIF (overall)
# -------------------------
# VIF expects a plain matrix; no constant needed for this
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns,
    name="VIF"
).sort_values(ascending=False)

print("\n=== VIF (sorted) ===")
print(vif.to_string())

# -------------------------
# 3) "Which variable causes the blow-up?" (drop-one check)
# -------------------------
# This is the simplest way to pinpoint the driver.
suspects = [c for c in X.columns if c.endswith("_between")]  # usually where issues live
print("\n=== Drop-one check (max VIF after removing each) ===")
for s in suspects:
    X2 = X.drop(columns=[s])
    v2 = max(variance_inflation_factor(X2.values, i) for i in range(X2.shape[1]))
    print(f"drop {s:30s} -> max VIF = {v2:8.2f}")

# Optional: focus only on between block correlations
between_cols = [c for c in X.columns if c.endswith("_between")]
if len(between_cols) >= 2:
    print("\n=== Between-only correlations (top |r|) ===")
    cb = X[between_cols].corr()
    abs_cb = cb.abs()
    np.fill_diagonal(abs_cb.values, np.nan)
    topb = abs_cb.stack().sort_values(ascending=False).head(20)
    for (a, b), _ in topb.items():
        print(f"{a}  ~  {b}:  r={cb.loc[a,b]:.3f}")

In [ ]:
## reduced to PCs only

# =========================
# STATE vs TRAIT GLM — CONTENT ONLY (cluster-robust)
# =========================
N_PCS_CONTENT = 5
OUTCOME_BIN = "high_any_item_eq3"
MECH_COLS = [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]

X_cols = []
for c in MECH_COLS:
    X_cols += [f"{c}_within", f"{c}_between"]

dfm = pm_day.copy()
y_glm = dfm[OUTCOME_BIN].astype(float).values
mask_glm = np.isfinite(y_glm)
for c in list(X_cols):
    if c in dfm.columns:
        mask_glm &= np.isfinite(dfm[c].values)

dfm = dfm.loc[mask_glm].copy()
y_glm = dfm[OUTCOME_BIN].astype(int).values
X_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")
groups_glm = dfm[PID_COL].astype(str).values

print(f"N = {len(dfm)} | participants = {dfm[PID_COL].nunique()}")
print(f"Positive cases: {y_glm.sum()} ({y_glm.mean():.3f})")
print(f"Events per variable: {y_glm.sum()}/{len(X_cols)} = {y_glm.sum()/len(X_cols):.1f}")
print("")

glm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())
res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})
print(res.summary())

rows = []
for c in MECH_COLS:
    for part in ["within", "between"]:
        name = f"{c}_{part}"
        if name in res.params.index:
            rows.append({
                "feature": c, "component": part,
                "beta": float(res.params[name]),
                "SE": float(res.bse[name]),
                "z": float(res.tvalues[name]),
                "p": float(res.pvalues[name]),
                "OR": float(np.exp(res.params[name])),
                "CI_lo": float(res.conf_int().loc[name, 0]),
                "CI_hi": float(res.conf_int().loc[name, 1]),
            })

tab = pd.DataFrame(rows)
tab.to_csv(os.path.join(OUT_DIR, "state_trait_glm_content.csv"), index=False)

print("")
for _, row in tab.iterrows():
    if row["p"] < .10:
        sig = "**" if row["p"] < .05 else "*"
        print(f"  {row['feature']}_{row['component']}: b={row['beta']:.3f} z={row['z']:.3f} p={row['p']:.3f} OR={row['OR']:.3f} {sig}")

In [ ]:
## check VIF again (looks better now)
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_check = dfm[X_cols].astype(float).values
vif = pd.DataFrame({
    "feature": X_cols,
    "VIF": [variance_inflation_factor(X_check, i) for i in range(X_check.shape[1])]
})
print(vif.to_string(index=False))
print(f"\nEvents per variable: {y_glm.sum()} events / {len(X_cols)} predictors = {y_glm.sum()/len(X_cols):.1f}")

In [ ]:
# now do the same but for crisis symptoms as the outcome

import numpy as np
import pandas as pd
import statsmodels.api as sm

CRISIS_COL = "crisis_PM_from_full"
PID_COL = "expiwell_id_clean"

PC_COLS = [
    "PC1_within","PC1_between",
    "PC2_within","PC2_between",
    "PC3_within","PC3_between",
    "PC4_within","PC4_between",
    "PC5_within","PC5_between",
]

dfm = pm_day.copy()

# mask: y present + pid present + all predictors present
y_raw = pd.to_numeric(dfm[CRISIS_COL], errors="coerce")
mask = y_raw.notna() & dfm[PID_COL].notna()

present = []
for c in PC_COLS:
    if c in dfm.columns:
        present.append(c)
        mask &= pd.to_numeric(dfm[c], errors="coerce").notna()
    else:
        print(f"[WARN] {c} missing; skipping")

dfm = dfm.loc[mask].copy()

y = pd.to_numeric(dfm[CRISIS_COL], errors="coerce").astype(float).values
X = dfm[present].apply(pd.to_numeric, errors="coerce").astype(float)
X = sm.add_constant(X, has_constant="add")
groups = dfm[PID_COL].astype(str).values

print(f"N = {len(dfm)} | participants = {dfm[PID_COL].nunique()}")
print(f"y mean={y.mean():.3f} sd={y.std():.3f} min={y.min():.3f} max={y.max():.3f}\n")

ols = sm.OLS(y, X)
res = ols.fit(cov_type="cluster", cov_kwds={"groups": groups})

print("=== STATE vs TRAIT OLS (crisis symptoms, cluster-robust) ===")
print(res.summary())

print("\n--- Significant / trending effects (p < .10) ---")
for term in X.columns:
    if term == "const":
        continue
    p = float(res.pvalues[term])
    if p < 0.10:
        sig = "**" if p < 0.05 else "*"
        b = float(res.params[term])
        t = float(res.tvalues[term])
        se = float(res.bse[term])
        print(f"  {term}: b={b:.3f} SE={se:.3f} t={t:.3f} p={p:.3f} {sig}")

In [ ]:
# how many zero days? zero inflated? 

(np.mean(y == 0), np.percentile(y, [0, 25, 50, 75, 100]))

In [ ]:
## try poisson GLM 

import numpy as np
import pandas as pd
import statsmodels.api as sm

CRISIS_COL = "crisis_PM_from_full"
PID_COL = "expiwell_id_clean"

PC_COLS = [
    "PC1_within","PC1_between",
    "PC2_within","PC2_between",
    "PC3_within","PC3_between",
    "PC4_within","PC4_between",
    "PC5_within","PC5_between",
]

dfm = pm_day.copy()

y_raw = pd.to_numeric(dfm[CRISIS_COL], errors="coerce")
mask = y_raw.notna() & dfm[PID_COL].notna()

present = []
for c in PC_COLS:
    if c in dfm.columns:
        present.append(c)
        mask &= pd.to_numeric(dfm[c], errors="coerce").notna()

dfm = dfm.loc[mask].copy()

y = pd.to_numeric(dfm[CRISIS_COL], errors="coerce").astype(float).values
X = dfm[present].apply(pd.to_numeric, errors="coerce").astype(float)
X = sm.add_constant(X, has_constant="add")
groups = dfm[PID_COL].astype(str).values

# ---- Poisson GLM (log link) ----
pois = sm.GLM(y, X, family=sm.families.Poisson())
res_pois = pois.fit(cov_type="cluster", cov_kwds={"groups": groups})

print("=== Poisson GLM (crisis symptoms, cluster-robust) ===")
print(res_pois.summary())

# quick overdispersion check (rule of thumb)
pearson_overdisp = res_pois.pearson_chi2 / res_pois.df_resid
print("\nOverdispersion (Pearson chi2 / df):", float(pearson_overdisp))

# If overdispersion is clearly > ~1.5–2, consider NegBin:
# ---- Negative Binomial GLM ----
nb = sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=1.0))
res_nb = nb.fit(cov_type="cluster", cov_kwds={"groups": groups})
print("\n=== NegBin GLM (alpha fixed=1.0; cluster-robust) ===")
print(res_nb.summary())

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

CRISIS_COL = "crisis_PM_from_full"
PID_COL = "expiwell_id_clean"

PC_COLS = [f"PC{i}_{part}" for i in range(1, 6) for part in ("within", "between")]
#NLP_COLS = [
#    "log1p_wc_within","log1p_wc_between",
#    "instability_cosdist_within","instability_cosdist_between",
#    "root_ttr_within","root_ttr_between",
#]
PRED_COLS = PC_COLS #+ NLP_COLS

# ---- build dfm, coerce numeric, drop missing ----
dfm = pm_day[[PID_COL, CRISIS_COL] + PRED_COLS].copy()

dfm[CRISIS_COL] = pd.to_numeric(dfm[CRISIS_COL], errors="coerce")
for c in PRED_COLS:
    dfm[c] = pd.to_numeric(dfm[c], errors="coerce")

dfm = dfm.dropna(subset=[PID_COL, CRISIS_COL] + PRED_COLS).copy()

y = dfm[CRISIS_COL].to_numpy(dtype=float)
groups = dfm[PID_COL].astype(str).to_numpy()

# IMPORTANT: make X a DataFrame so params are labeled
X_df = sm.add_constant(dfm[PRED_COLS].astype(float), has_constant="add")

print(f"N={len(dfm)} | participants={dfm[PID_COL].nunique()} | predictors={X_df.shape[1]-1}")
assert len(y) == X_df.shape[0] == len(groups)

# ---- Poisson -> alpha_hat ----
pois = sm.GLM(y, X_df, family=sm.families.Poisson())
res_pois = pois.fit(cov_type="cluster", cov_kwds={"groups": groups})
alpha_hat = max((res_pois.pearson_chi2 / res_pois.df_resid) - 1.0, 1e-6)
print("alpha_hat (MoM-ish):", float(alpha_hat))

# ---- NegBin with cluster-robust ----
nb = sm.GLM(y, X_df, family=sm.families.NegativeBinomial(alpha=alpha_hat))
res_nb = nb.fit(cov_type="cluster", cov_kwds={"groups": groups})

print("\n=== Negative Binomial GLM (cluster-robust) ===")
print(res_nb.summary())

# ---- per-SD multiplicative effects exp(beta*SD) ----
print("\n--- Per-SD multiplicative effects exp(beta*SD) ---")
sds = dfm[PRED_COLS].std()
for term in PRED_COLS:
    b = float(res_nb.params[term])         # now works (params is a Series)
    sd = float(sds[term])
    mult_1sd = float(np.exp(b * sd))
    print(f"{term:28s} beta={b: .3f}  SD={sd:.3f}  exp(beta*SD)={mult_1sd:.3f}")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

CRISIS_COL = "crisis_PM_from_full"
PID_COL = "expiwell_id_clean"

# PCs (within + between)
PC_COLS = [f"PC{i}_{part}" for i in range(1, 6) for part in ("within", "between")]

# Other NLP features (within + between)
NLP_COLS = [
    "log1p_wc_within",
    "log1p_wc_between",
    "instability_cosdist_within",
    "instability_cosdist_between",
    "root_ttr_within",
    "root_ttr_between",
]

PRED_COLS = PC_COLS + NLP_COLS

dfm = pm_day.copy()

# ---- complete-case mask for y, pid, predictors ----
y_raw = pd.to_numeric(dfm[CRISIS_COL], errors="coerce")
mask = y_raw.notna() & dfm[PID_COL].notna()

present = []
for c in PRED_COLS:
    if c in dfm.columns:
        present.append(c)
        mask &= pd.to_numeric(dfm[c], errors="coerce").notna()
    else:
        print(f"[WARN] missing predictor: {c} (skipping)")

dfm = dfm.loc[mask].copy()

y = pd.to_numeric(dfm[CRISIS_COL], errors="coerce").astype(float).values
X = dfm[present].apply(pd.to_numeric, errors="coerce").astype(float)
X = sm.add_constant(X, has_constant="add")
groups = dfm[PID_COL].astype(str).values

print(f"N={len(dfm)} | participants={dfm[PID_COL].nunique()} | predictors={X.shape[1]-1}")

# ---- quick collinearity diagnostics (optional but recommended) ----
between_cols = [c for c in present if c.endswith("_between")]
if "log1p_wc_between" in between_cols and "root_ttr_between" in between_cols:
    r = dfm[["log1p_wc_between", "root_ttr_between"]].corr().iloc[0,1]
    print("corr(log1p_wc_between, root_ttr_between) =", float(r))

# VIF (not cluster-aware; just to spot redundancy)
try:
    X_no_const = X.drop(columns=["const"], errors="ignore")
    vifs = pd.Series(
        [variance_inflation_factor(X_no_const.values, i) for i in range(X_no_const.shape[1])],
        index=X_no_const.columns
    ).sort_values(ascending=False)
    print("\nTop VIFs:")
    print(vifs.head(10).to_string())
except Exception as e:
    print("[WARN] VIF failed:", e)

# ---- estimate alpha via Poisson overdispersion ----
pois = sm.GLM(y, X, family=sm.families.Poisson())
res_pois = pois.fit(cov_type="cluster", cov_kwds={"groups": groups})
alpha_hat = max((res_pois.pearson_chi2 / res_pois.df_resid) - 1.0, 1e-6)
print("\nalpha_hat:", float(alpha_hat))

# ---- NegBin GLM with alpha_hat + cluster robust ----
nb = sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=alpha_hat))
res_nb = nb.fit(cov_type="cluster", cov_kwds={"groups": groups})

print("\n=== NB GLM (PCs + NLP, cluster-robust) ===")
print(res_nb.summary())

# ---- per-SD multipliers for interpretability ----
print("\n--- Per-SD multiplicative effects exp(beta*SD) (p < .10) ---")
sds = X.drop(columns=["const"]).std()
for term in sds.index:
    b = float(res_nb.params[term])
    mult = float(np.exp(b * sds[term]))
    p = float(res_nb.pvalues[term])
    if p < 0.10:
        sig = "**" if p < 0.05 else "*"
        print(f"{term:30s} exp(beta*SD)={mult: .3f}  p={p:.3g} {sig}")

In [ ]:
import pandas as pd
import numpy as np

CRISIS_COL = "crisis_PM_from_full"

cols = [
    CRISIS_COL,
    "PC1_within","PC1_between",
    "PC2_within","PC2_between",
    "PC3_within","PC3_between",
    "PC4_within","PC4_between",
    "PC5_within","PC5_between",
    "log1p_wc_within","log1p_wc_between",
    "instability_cosdist_within","instability_cosdist_between",
    "root_ttr_within","root_ttr_between"
]

# keep only columns that exist
cols_present = [c for c in cols if c in pm_day.columns]
missing = [c for c in cols if c not in pm_day.columns]
if missing:
    print("[WARN] missing columns:", missing)

dfc = pm_day[cols_present].apply(pd.to_numeric, errors="coerce")

# pairwise-complete Pearson correlations
corr = dfc.corr(method="pearson")

# display nicely (rounded)
print(corr.round(3).to_string())

In [ ]:
import numpy as np
import pandas as pd

CRISIS_COL = "crisis_PM_from_full"

cols = [
    CRISIS_COL,
    "PC1_within","PC1_between","PC2_within","PC2_between","PC3_within","PC3_between",
    "PC4_within","PC4_between","PC5_within","PC5_between",
    "log1p_wc_within","log1p_wc_between",
    "instability_cosdist_within","instability_cosdist_between",
    "root_ttr_within","root_ttr_between"
]

dfc = pm_day[[c for c in cols if c in pm_day.columns]].apply(pd.to_numeric, errors="coerce")
corr = dfc.corr()

# crisis correlations (sorted by |r|)
cr = corr[CRISIS_COL].drop(CRISIS_COL).sort_values(key=lambda s: s.abs(), ascending=False)
print("\n--- Corr with crisis total (sorted by |r|) ---")
print(cr.round(3).to_string())

# top redundant predictor pairs (|r| >= .70)
pairs = []
c = corr.drop(index=[CRISIS_COL], columns=[CRISIS_COL])
for i in range(len(c.columns)):
    for j in range(i+1, len(c.columns)):
        r = c.iloc[i, j]
        if abs(r) >= 0.70:
            pairs.append((c.columns[i], c.columns[j], r))

pairs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
print("\n--- Highly correlated predictor pairs (|r| >= .70) ---")
for a,b,r in pairs[:20]:
    print(f"{a:28s} ~ {b:28s} r={r:.3f}")

In [ ]:
import statsmodels.api as sm
df = pm_day[["PC2_between","log1p_wc_between"]].dropna()
print(sm.OLS(df["PC2_between"], sm.add_constant(df["log1p_wc_between"])).fit().summary())

In [ ]:
import statsmodels.api as sm
y = pd.to_numeric(pm_day["crisis_PM_from_full"], errors="coerce")
x = pd.to_numeric(pm_day["log1p_wc_within"], errors="coerce")
mask = y.notna() & x.notna() & pm_day["expiwell_id_clean"].notna()
df = pm_day.loc[mask].copy()

X = sm.add_constant(df["log1p_wc_within"].astype(float))
y = df["crisis_PM_from_full"].astype(float).values
g = df["expiwell_id_clean"].astype(str).values

res = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": g})
print(res.summary())